# 03 - Feature Engineering Experiments

This notebook experiments with different feature engineering approaches to improve gesture recognition.

## Features to Explore

**In SRC (Production):**
1. **Relative Positions** - Landmark positions relative to wrist (translation invariant)
2. **Velocity** - Frame-to-frame position changes (for Index MCP & Tip)
3. **Finger Angles** - 5 bending angles (1 per finger at middle joint)
4. **Inter-finger Distances** - Thumb to all fingertips (5 distances)

**Experimental (Not in SRC):**
5. **Acceleration** - Rate of velocity change
6. **Hand Bounding Box** - Size and aspect ratio for distance awareness
7. **Finger Curl** - Tip-to-wrist distances (fist/open detection)
8. **Palm Orientation** - Palm normal vector
9. **15 Finger Angles** - All 15 joint angles (3 per finger)

In [ ]:
# Imports
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from handflow.features import FeatureEngineer

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Imports loaded")

In [ ]:
#Data paths
DATA_PATH = Path('../data/raw/right_mp_Data')
if not DATA_PATH.exists():
    DATA_PATH = Path('../ModelTraining/MP_Data')

ACTIONS = ['none', 'horizontal_swipe', 'zoom', 'pointyclick', 'middleclick', 'swipeup', 'swipedown', 'thumb_left', 'thumb_right', 'touch', 'touch_hold', 'touch_hover']
print(f"📁 Data path: {DATA_PATH}")
print(f"🎯 Actions: {ACTIONS}")

In [ ]:
from pathlib import Path
import numpy as np

def load_raw_data(
    data_path,
    actions,
    sequence_length=12,
    sequences_per_class=50,
):
    """
    Load raw data with a fixed number of sequences per class.

    Returns:
        sequences: (N, T, F)
        labels:    (N,) integer class ids
        label_map: dict[str, int]
    """

    data_path = Path(data_path)
    sequences = []
    labels = []

    label_map = {action: i for i, action in enumerate(actions)}

    for action in actions:
        action_path = data_path / action
        if not action_path.exists():
            print(f"⚠️ Missing: {action_path}")
            continue

        seq_dirs = sorted(d for d in action_path.iterdir() if d.is_dir())
        loaded = 0

        for seq_dir in seq_dirs:
            if loaded >= sequences_per_class:
                break

            window = []
            for frame_num in range(sequence_length):
                frame_path = seq_dir / f"{frame_num}.npy"
                if not frame_path.exists():
                    break
                window.append(np.load(frame_path))

            if len(window) == sequence_length:
                sequences.append(window)
                labels.append(label_map[action])
                loaded += 1

        print(f"✅ Loaded {loaded}/{sequences_per_class} sequences for '{action}'")

    return np.asarray(sequences), np.asarray(labels), label_map


In [ ]:
sequences, labels, label_map = load_raw_data(
    DATA_PATH,
    ACTIONS,
    sequence_length=12,
    sequences_per_class=50
)

print(sequences.shape)  # (num_classes * 50, 12, 84)


In [ ]:
# Create gesture samples dictionary for visualization
# Pick one sample per gesture class for plotting
inverse_label_map = {v: k for k, v in label_map.items()}
gesture_samples = {}

for label_id in np.unique(labels):
    gesture_name = inverse_label_map[label_id]
    # Get first sequence for this label
    idx = np.where(labels == label_id)[0][0]
    gesture_samples[gesture_name] = sequences[idx]

print(f"✅ Created gesture_samples with {len(gesture_samples)} gestures")
print(f"   Gestures: {list(gesture_samples.keys())}")

# Initialize the FeatureEngineer from src
engineer = FeatureEngineer()
print(f"✅ Initialized FeatureEngineer (output dim: {engineer.get_output_dim()})")

## 1. Velocity Features

In [ ]:
import numpy as np

def compute_velocity(sequence):
    return np.diff(sequence, axis=0, prepend=sequence[:1])


# Test on single sequence
sample_seq = sequences[1]
sample_velocity = compute_velocity(sample_seq)

print(f"Original shape: {sample_seq.shape}")
print(f"Velocity shape: {sample_velocity.shape}")
print(f"Velocity range: [{sample_velocity.min():.4f}, {sample_velocity.max():.4f}]")

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

# ---- dynamic layout ----
N = len(gesture_samples)
n_cols = 2
n_rows = math.ceil(N / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(7 * n_cols, 4 * n_rows),
    squeeze=False
)

axes = axes.flatten()

# ---- plot velocity curves ----
for idx, (gesture, seq) in enumerate(gesture_samples.items()):
    if seq is None:
        continue

    ax = axes[idx]

    T = seq.shape[0]  # number of frames

    # Compute per-frame velocity
    velocity = compute_velocity(seq)  # (T, 84)

    # Reshape to (frames, landmarks, xyz+visibility)
    velocity_xyz = velocity.reshape(T, 21, 4)[:, :, :3]

    # Compute magnitude for each landmark
    velocity_mag = np.linalg.norm(velocity_xyz, axis=2)  # (T, 21)

    # Plot each landmark as a line
    for landmark_idx in range(velocity_mag.shape[1]):
        ax.plot(
            range(T),
            velocity_mag[:, landmark_idx],
            alpha=0.7
        )

    ax.set_title(f"{gesture} - Landmark Velocities")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Velocity Magnitude")
    ax.grid(True)

# ---- remove unused axes ----
for i in range(idx + 1, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()


## 2. Acceleration Features (Experimental)

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

def compute_acceleration(sequence):
    """
    Compute frame-to-frame acceleration (rate of velocity change).
    Returns:
        acceleration: (T, 84) array
    """
    velocity = compute_velocity(sequence)
    acceleration = np.diff(velocity, axis=0, prepend=velocity[0:1])
    return acceleration

# ---- filter valid gestures ----
valid_gestures = [(g, s) for g, s in gesture_samples.items() if s is not None]
N = len(valid_gestures)

fig, axes = plt.subplots(
    N,
    2,
    figsize=(14, 4 * N),
    squeeze=False
)

# ---- plotting ----
for row, (gesture, sample) in enumerate(valid_gestures):

    T = sample.shape[0]
    velocity = compute_velocity(sample)         # (T, 84)
    acceleration = compute_acceleration(sample) # (T, 84)

    # reshape: (frames, landmarks, xyz)
    velocity_xyz = velocity.reshape(T, 21, 4)[:, :, :3]
    acceleration_xyz = acceleration.reshape(T, 21, 4)[:, :, :3]

    # magnitude per landmark
    vel_mag = np.linalg.norm(velocity_xyz, axis=2)        # (T, 21)
    acc_mag = np.linalg.norm(acceleration_xyz, axis=2)   # (T, 21)

    # --- Time-series plot per landmark ---
    ax_ts = axes[row, 0]
    for landmark_idx in range(vel_mag.shape[1]):
        ax_ts.plot(range(T), vel_mag[:, landmark_idx], alpha=0.7)
        ax_ts.plot(range(T), acc_mag[:, landmark_idx], alpha=0.5, linestyle='--')

    ax_ts.set_xlabel('Frame')
    ax_ts.set_ylabel('Magnitude')
    ax_ts.set_title(f'{gesture} — Velocity vs Acceleration')
    ax_ts.grid(True)

    # optional: single legend
    ax_ts.plot([], [], color='blue', label='Velocity')
    ax_ts.plot([], [], color='orange', linestyle='--', label='Acceleration')
    ax_ts.legend(loc='upper right')

    # --- Scatter plot per landmark ---
    ax_scatter = axes[row, 1]
    for landmark_idx in range(vel_mag.shape[1]):
        ax_scatter.scatter(vel_mag[:, landmark_idx], acc_mag[:, landmark_idx], alpha=0.7)

    ax_scatter.set_xlabel('Velocity Magnitude')
    ax_scatter.set_ylabel('Acceleration Magnitude')
    ax_scatter.set_title('Velocity–Acceleration Correlation')
    ax_scatter.grid(True)

plt.tight_layout()
plt.show()


## 3. Finger Angle Features

In [ ]:
# SRC-COMPATIBLE: 5 finger angles (1 per finger at middle joint)
# This matches src/handflow/features/feature_engineer.py

def compute_finger_angles_src(sequence):
    """
    Compute bending angles for all 5 fingers (matches src implementation).
    Angle is computed at the middle joint (PIP for fingers, IP for thumb).
    
    Returns: 
        angles: (T, 5) where values are in radians
    """
    # Indices for joints [Proximal, Middle, Distal]
    joints = [
        (2, 3, 4),    # Thumb: CMC -> IP -> Tip
        (5, 6, 7),    # Index: MCP -> PIP -> DIP
        (9, 10, 11),  # Middle: MCP -> PIP -> DIP
        (13, 14, 15), # Ring: MCP -> PIP -> DIP
        (17, 18, 19)  # Pinky: MCP -> PIP -> DIP
    ]
    
    T = sequence.shape[0]
    angles = np.zeros((T, 5))
    
    for i, (a_idx, b_idx, c_idx) in enumerate(joints):
        # Extract points (T, 3)
        a = sequence[:, a_idx*4 : a_idx*4+3]
        b = sequence[:, b_idx*4 : b_idx*4+3]
        c = sequence[:, c_idx*4 : c_idx*4+3]
        
        # Vectors BA and BC
        ba = a - b
        bc = c - b
        
        # Compute angle
        norm_ba = np.linalg.norm(ba, axis=1, keepdims=True) + 1e-8
        norm_bc = np.linalg.norm(bc, axis=1, keepdims=True) + 1e-8
        
        dot_prod = np.sum(ba * bc, axis=1, keepdims=True) / (norm_ba * norm_bc)
        dot_prod = np.clip(dot_prod, -1.0, 1.0)
        
        angle_rad = np.arccos(dot_prod)
        angles[:, i] = angle_rad.flatten()
        
    return angles

# Test
sample_angles_src = compute_finger_angles_src(sequences[0])
print(f"SRC Angles shape: {sample_angles_src.shape}")
print(f"SRC Angle range (radians): [{sample_angles_src.min():.4f}, {sample_angles_src.max():.4f}]")
print(f"SRC Angle range (degrees): [{np.degrees(sample_angles_src.min()):.1f}, {np.degrees(sample_angles_src.max()):.1f}]")

In [ ]:
# Finger joint indices for angle calculation
FINGER_JOINTS = [
    # Thumb
    (0, 1, 2), (1, 2, 3), (2, 3, 4),
    # Index
    (0, 5, 6), (5, 6, 7), (6, 7, 8),
    # Middle
    (0, 9, 10), (9, 10, 11), (10, 11, 12),
    # Ring
    (0, 13, 14), (13, 14, 15), (14, 15, 16),
    # Pinky
    (0, 17, 18), (17, 18, 19), (18, 19, 20),
]

def calculate_angle(p1, p2, p3):
    """
    Calculate angle at p2 formed by p1-p2-p3.
    
    Returns:
        angle in radians normalized to [0, 1]
    """
    v1 = p1 - p2
    v2 = p3 - p2
    
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    angle = np.arccos(cos_angle)
    
    return angle / np.pi  # Normalize to [0, 1]

def compute_finger_angles(sequence):
    """
    Compute all 15 finger joint angles for each frame.
    
    Returns:
        angles: (16, 15) array
    """
    num_frames = sequence.shape[0]
    angles = np.zeros((num_frames, 15))
    
    for frame_idx in range(num_frames):
        landmarks = sequence[frame_idx].reshape(21, 4)[:, :3]  # x, y, z only
        
        for joint_idx, (p1, p2, p3) in enumerate(FINGER_JOINTS):
            angles[frame_idx, joint_idx] = calculate_angle(
                landmarks[p1], landmarks[p2], landmarks[p3]
            )
    
    return angles

# Test
sample_angles = compute_finger_angles(sequences[0])
print(f"Angles shape: {sample_angles.shape}")
print(f"Angle range: [{sample_angles.min():.4f}, {sample_angles.max():.4f}]")

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

# ---- filter valid gestures ----
valid_gestures = [(g, s) for g, s in gesture_samples.items() if s is not None]
N = len(valid_gestures)

n_cols = 2
n_rows = math.ceil(N / n_cols)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(7 * n_cols, 4 * n_rows),
    squeeze=False
)
axes = axes.flatten()

# ---- constants ----
finger_names = ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']
joint_per_finger = 3

colors_map = {
    'Thumb': 'red',
    'Index': 'orange',
    'Middle': 'green',
    'Ring': 'blue',
    'Pinky': 'purple'
}

# ---- plotting ----
for idx, (gesture, seq) in enumerate(valid_gestures):
    ax = axes[idx]

    # Compute finger angles: shape (T, 15)
    angles = compute_finger_angles(seq)  

    # Average over frames
    avg_angles = angles.mean(axis=0)  # (15,)

    # Build colors per joint
    colors = []
    labels = []
    for i, finger in enumerate(finger_names):
        for j in range(joint_per_finger):
            colors.append(colors_map[finger])
            labels.append(finger)

    # Bar plot
    ax.bar(range(15), avg_angles, color=colors)
    ax.set_xlabel('Joint Index')
    ax.set_ylabel('Angle (normalized)')
    ax.set_title(f'{gesture} — Average Finger Angles')
    ax.set_ylim(0, 1)

    # Optional: label fingers on x-axis
    ax.set_xticks(range(15))
    ax.set_xticklabels(labels, rotation=45, ha='right')

# ---- remove unused axes ----
for i in range(idx + 1, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.savefig('../docs/finger_angles.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Hand Bounding Box Features (Experimental)

In [ ]:
def compute_bbox_features(sequence):
    """
    Compute hand bounding box features.
    
    Features:
    - width: horizontal span
    - height: vertical span
    - area: width * height (indicates distance from camera)
    - aspect_ratio: width / height
    
    Returns:
        bbox_features: (16, 4) array
    """
    num_frames = sequence.shape[0]
    bbox_features = np.zeros((num_frames, 4))
    
    for frame_idx in range(num_frames):
        landmarks = sequence[frame_idx].reshape(21, 4)
        
        x_coords = landmarks[:, 0]
        y_coords = landmarks[:, 1]
        
        width = x_coords.max() - x_coords.min()
        height = y_coords.max() - y_coords.min()
        area = width * height
        aspect_ratio = width / (height + 1e-8)
        
        bbox_features[frame_idx] = [width, height, area, aspect_ratio]
    
    return bbox_features

# Test
sample_bbox = compute_bbox_features(sequences[0])
print(f"Bbox features shape: {sample_bbox.shape}")
print(f"Width range: [{sample_bbox[:, 0].min():.4f}, {sample_bbox[:, 0].max():.4f}]")

In [ ]:
import matplotlib.pyplot as plt

# ---- layout: one subplot per bbox feature ----
feature_names = ['Width', 'Height', 'Area', 'Aspect Ratio']
n_features = len(feature_names)

n_cols = 2
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(7 * n_cols, 4 * n_rows),
    squeeze=False
)

axes = axes.flatten()

# ---- plot all gestures ----
for gesture, seq in gesture_samples.items():
    if seq is None:
        continue

    bbox = compute_bbox_features(seq)  # (T, 4)

    for feat_idx, feat_name in enumerate(feature_names):
        axes[feat_idx].plot(
            bbox[:, feat_idx],
            label=gesture,
            alpha=0.8
        )

# ---- axis formatting ----
for feat_idx, feat_name in enumerate(feature_names):
    ax = axes[feat_idx]
    ax.set_xlabel('Frame')
    ax.set_ylabel(feat_name)
    ax.set_title(f'Hand {feat_name} Over Time')

# ---- figure-level horizontal legend on top ----
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.05),
    ncol=len(labels),  # horizontal
    frameon=False      # optional: no box
)

# ---- remove unused axes ----
for i in range(n_features, len(axes)):
    fig.delaxes(axes[i])

# ---- layout and save ----
plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space for legend
plt.savefig('../docs/bbox_features.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Inter-finger Distances (Pinch Detection)

In [ ]:
def compute_inter_finger_distances(sequence):
    """
    Compute distances from thumb to all fingertips (matches src/handflow/features/feature_engineer.py).
    
    Pairs:
    - Thumb tip (4) to Index tip (8)
    - Thumb tip (4) to Middle tip (12)  
    - Thumb tip (4) to Ring tip (16)
    - Thumb tip (4) to Pinky tip (20)
    - Thumb tip (4) to Index PIP (6)

    Returns:
        distances: (T, 5)
    """
    num_frames = sequence.shape[0]
    distances = np.zeros((num_frames, 5))

    # Thumb to all fingertips + Index PIP (matches src)
    pairs = [(4, 8), (4, 12), (4, 16), (4, 20), (4, 6)]

    for i, (p1, p2) in enumerate(pairs):
        p1_coords = sequence[:, p1*4 : p1*4 + 3]
        p2_coords = sequence[:, p2*4 : p2*4 + 3]
        distances[:, i] = np.linalg.norm(p1_coords - p2_coords, axis=1)

    return distances

# Test
sample_dist = compute_inter_finger_distances(sequences[0])
print(f"Inter-finger distances shape: {sample_dist.shape}")
print(f"Distance range: [{sample_dist.min():.4f}, {sample_dist.max():.4f}]")

In [ ]:
import matplotlib.pyplot as plt

distance_names = [
    'Thumb-Index Tip',
    'Thumb-Middle Tip',
    'Thumb-Ring Tip',
    'Thumb-Pinky Tip',
    'Thumb-Index PIP'
]

n_features = len(distance_names)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(6 * n_cols, 4 * n_rows),
    squeeze=False
)

axes = axes.flatten()

# ---- plot all gestures ----
for gesture, seq in gesture_samples.items():
    if seq is None:
        continue

    distances = compute_inter_finger_distances(seq)  # (T, 5)

    for feat_idx, feat_name in enumerate(distance_names):
        axes[feat_idx].plot(
            distances[:, feat_idx],
            label=gesture,
            alpha=0.8
        )

# ---- formatting ----
for feat_idx, feat_name in enumerate(distance_names):
    ax = axes[feat_idx]
    ax.set_xlabel('Frame')
    ax.set_ylabel('Distance')
    ax.set_title(f'{feat_name} Distance Over Time')

# ---- figure-level legend ----
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.02),
    ncol=min(len(labels), 6),
    frameon=False
)

# ---- remove unused axes ----
for i in range(n_features, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig('../docs/inter_finger_distances.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Finger Curl (Experimental)

Tip-to-wrist distances - useful for detecting fist/open hand gestures.

In [ ]:
def compute_finger_curl(sequence):
    """
    Compute distance from finger tips to wrist (Landmark 0)
    """
    num_frames = sequence.shape[0]
    curls = np.zeros((num_frames, 5))

    tips = [4, 8, 12, 16, 20]
    wrist = sequence[:, 0:3] 
    
    for i, tip in enumerate(tips): 
        tip_coords = sequence[:, tip*4 : tip*4+3]
        curls[:, i] = np.linalg.norm(tip_coords - wrist, axis=1)
        
    return curls

In [ ]:
import matplotlib.pyplot as plt

finger_names = ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']
n_features = len(finger_names)

n_cols = 2
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(7 * n_cols, 4 * n_rows),
    squeeze=False
)

axes = axes.flatten()

# ---- plot all gestures ----
for gesture, seq in gesture_samples.items():
    if seq is None:
        continue

    curls = compute_finger_curl(seq)  # (T, 5)

    for feat_idx, finger in enumerate(finger_names):
        axes[feat_idx].plot(
            curls[:, feat_idx],
            label=gesture,
            alpha=0.8
        )

# ---- formatting ----
for feat_idx, finger in enumerate(finger_names):
    ax = axes[feat_idx]
    ax.set_xlabel('Frame')
    ax.set_ylabel('Tip–Wrist Distance')
    ax.set_title(f'{finger} Finger Curl Over Time')
    ax.legend()

# ---- remove unused axes ----
for i in range(n_features, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.savefig('../docs/finger_curl.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Palm Orientation (Experimental)

Palm normal vector via cross product - useful for detecting hand rotation.

In [ ]:
def compute_palm_orientation(sequence: np.ndarray) -> np.ndarray:
    """
    Compute palm normal vector via cross product of:
    Wrist -> Index MCP  and  Wrist -> Pinky MCP.

    Args:
        sequence: np.ndarray of shape (T, 21*4)
                  Each landmark = (x, y, z, visibility)

    Returns:
        normals: np.ndarray of shape (T, 3)
    """
    wrist = sequence[:, 0:3]                     # Landmark 0
    index_mcp = sequence[:, 5*4 : 5*4 + 3]       # Landmark 5
    pinky_mcp = sequence[:, 17*4 : 17*4 + 3]     # Landmark 17

    v1 = index_mcp - wrist
    v2 = pinky_mcp - wrist

    cross = np.cross(v1, v2)
    norms = np.linalg.norm(cross, axis=1, keepdims=True)

    normals = cross / (norms + 1e-8)
    return normals


In [ ]:
import matplotlib.pyplot as plt

component_names = ['Normal X', 'Normal Y', 'Normal Z']
n_features = len(component_names)

n_cols = 2
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(7 * n_cols, 4 * n_rows),
    squeeze=False
)

axes = axes.flatten()

# ---- plot all gestures ----
for gesture, seq in gesture_samples.items():
    if seq is None:
        continue

    normals = compute_palm_orientation(seq)  # (T, 3)

    for feat_idx, comp in enumerate(component_names):
        axes[feat_idx].plot(
            normals[:, feat_idx],
            label=gesture,
            alpha=0.8
        )

# ---- formatting ----
for feat_idx, comp in enumerate(component_names):
    ax = axes[feat_idx]
    ax.set_xlabel('Frame')
    ax.set_ylabel(comp)
    ax.set_title(f'Palm Orientation — {comp} Over Time')
    ax.legend()

# ---- remove unused axes ----
for i in range(n_features, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.savefig('../docs/palm_orientation.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Feature Set Comparison & Evaluation

Compare different feature combinations using a simple classifier to understand which features contribute most to gesture recognition accuracy.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

def prepare_features(sequences, feature_func, flatten=True):
    """Apply feature function to all sequences and optionally flatten."""
    features = np.array([feature_func(seq) for seq in sequences])
    if flatten:
        return features.reshape(len(sequences), -1)
    return features

def evaluate_features(X, y, name, cv=5):
    """Evaluate features using cross-validation with Random Forest."""
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ])
    
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X, y, cv=skf, scoring='accuracy')
    
    return {
        'name': name,
        'mean_acc': scores.mean(),
        'std_acc': scores.std(),
        'n_features': X.shape[1]
    }

print("✅ Evaluation functions defined")

In [ ]:
# Define feature extraction functions for different feature sets

def extract_raw_positions(seq):
    """Raw keypoints flattened."""
    return seq.reshape(seq.shape[0], -1)

def extract_relative_positions(seq):
    """Positions relative to wrist (translation invariant)."""
    landmarks_xyz = seq.reshape(-1, 21, 4)[:, :, :3]
    wrist = landmarks_xyz[:, 0:1, :]
    relative = landmarks_xyz - wrist
    return relative.reshape(seq.shape[0], -1)

def extract_velocity_only(seq):
    """Velocity features only."""
    return compute_velocity(seq)

def extract_src_features(seq):
    """Features matching src/handflow/features/feature_engineer.py"""
    return engineer.transform(seq)

def extract_experimental_all(seq):
    """All experimental features combined."""
    T = seq.shape[0]
    
    # Relative positions (63)
    landmarks_xyz = seq.reshape(-1, 21, 4)[:, :, :3]
    wrist = landmarks_xyz[:, 0:1, :]
    relative = (landmarks_xyz - wrist).reshape(T, -1)
    
    # Velocity (84)
    velocity = compute_velocity(seq)
    
    # Acceleration (84)
    acceleration = compute_acceleration(seq)
    
    # 15 finger angles (experimental)
    angles_15 = compute_finger_angles(seq)
    
    # 5 finger angles (src-compatible)
    angles_5 = compute_finger_angles_src(seq)
    
    # Bbox features (4)
    bbox = compute_bbox_features(seq)
    
    # Inter-finger distances (5)
    distances = compute_inter_finger_distances(seq)
    
    # Finger curl (5)
    curls = compute_finger_curl(seq)
    
    # Palm orientation (3)
    palm = compute_palm_orientation(seq)
    
    return np.concatenate([
        relative,      # 63
        velocity,      # 84
        acceleration,  # 84
        angles_15,     # 15
        angles_5,      # 5
        bbox,          # 4
        distances,     # 5
        curls,         # 5
        palm           # 3
    ], axis=-1)

print("✅ Feature extraction functions defined")

In [ ]:
# Evaluate all feature sets
print("🔄 Evaluating feature sets (this may take a moment)...")
print()

feature_sets = [
    ("Raw Keypoints (84)", extract_raw_positions),
    ("Relative Positions (63)", extract_relative_positions),
    ("Velocity Only (84)", extract_velocity_only),
    ("SRC Features (88)", extract_src_features),
    ("All Experimental", extract_experimental_all),
]

results = []
for name, func in feature_sets:
    print(f"  Evaluating: {name}...", end=" ")
    X = prepare_features(sequences, func)
    result = evaluate_features(X, labels, name)
    results.append(result)
    print(f"Acc: {result['mean_acc']:.3f} ± {result['std_acc']:.3f} ({result['n_features']} features)")

print()
print("✅ Evaluation complete!")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of accuracy
names = [r['name'] for r in results]
accs = [r['mean_acc'] for r in results]
stds = [r['std_acc'] for r in results]
n_feats = [r['n_features'] for r in results]

colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(results)))

ax1 = axes[0]
bars = ax1.bar(range(len(results)), accs, yerr=stds, capsize=5, color=colors, edgecolor='black')
ax1.set_xticks(range(len(results)))
ax1.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=45, ha='right')
ax1.set_ylabel('Accuracy')
ax1.set_title('Feature Set Comparison - Cross-Validation Accuracy')
ax1.set_ylim(0, 1)

# Add accuracy values on bars
for i, (bar, acc) in enumerate(zip(bars, accs)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{acc:.1%}', ha='center', va='bottom', fontsize=9)

# Scatter plot: features vs accuracy
ax2 = axes[1]
for i, (name, acc, n_feat) in enumerate(zip(names, accs, n_feats)):
    ax2.scatter(n_feat, acc, s=150, c=[colors[i]], label=name.split('(')[0].strip(), 
                edgecolor='black', linewidth=1)

ax2.set_xlabel('Number of Features')
ax2.set_ylabel('Accuracy')
ax2.set_title('Features vs Accuracy Trade-off')
ax2.legend(loc='lower right', fontsize=8)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../docs/feature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed results table
print("=" * 70)
print("📊 FEATURE SET COMPARISON RESULTS")
print("=" * 70)
print(f"{'Feature Set':<30} {'Accuracy':>12} {'Std Dev':>10} {'Features':>10}")
print("-" * 70)

# Sort by accuracy
sorted_results = sorted(results, key=lambda x: x['mean_acc'], reverse=True)

for r in sorted_results:
    print(f"{r['name']:<30} {r['mean_acc']:>11.1%} {r['std_acc']:>10.3f} {r['n_features']:>10}")

print("-" * 70)

# Find best
best = sorted_results[0]
print(f"\n🏆 Best: {best['name']} with {best['mean_acc']:.1%} accuracy")

# Compare SRC vs experimental
src_result = next(r for r in results if 'SRC' in r['name'])
exp_result = next(r for r in results if 'Experimental' in r['name'])

print(f"\n📈 SRC Features vs All Experimental:")
print(f"   SRC: {src_result['mean_acc']:.1%} ({src_result['n_features']} features)")
print(f"   Experimental: {exp_result['mean_acc']:.1%} ({exp_result['n_features']} features)")
print(f"   Difference: {(exp_result['mean_acc'] - src_result['mean_acc'])*100:+.1f}%")
print("=" * 70)

## 9. Individual Feature Importance Analysis

Analyze which individual feature types contribute most to the classification.

In [ ]:
# Evaluate individual feature types
print("🔄 Evaluating individual feature types...")
print()

individual_features = [
    ("Relative Positions", lambda s: (s.reshape(-1, 21, 4)[:, :, :3] - s.reshape(-1, 21, 4)[:, 0:1, :3]).reshape(s.shape[0], -1)),
    ("Velocity", compute_velocity),
    ("Acceleration", compute_acceleration),
    ("15 Finger Angles (Exp)", compute_finger_angles),
    ("5 Finger Angles (SRC)", compute_finger_angles_src),
    ("Bbox Features (Exp)", compute_bbox_features),
    ("Inter-finger Distances", compute_inter_finger_distances),
    ("Finger Curl (Exp)", compute_finger_curl),
    ("Palm Orientation (Exp)", compute_palm_orientation),
]

individual_results = []
for name, func in individual_features:
    print(f"  Evaluating: {name}...", end=" ")
    X = prepare_features(sequences, func)
    result = evaluate_features(X, labels, name)
    individual_results.append(result)
    print(f"Acc: {result['mean_acc']:.3f} ({result['n_features']} features)")

print()
print("✅ Individual evaluation complete!")

In [ ]:
# Visualize individual feature importance
fig, ax = plt.subplots(figsize=(12, 6))

# Sort by accuracy
sorted_ind = sorted(individual_results, key=lambda x: x['mean_acc'], reverse=True)
names = [r['name'] for r in sorted_ind]
accs = [r['mean_acc'] for r in sorted_ind]
stds = [r['std_acc'] for r in sorted_ind]

# Color by whether it's in SRC or experimental
colors = ['#2ecc71' if '(SRC)' in n or 'Positions' in n or 'Velocity' in n or 'Inter-finger' in n 
          else '#e74c3c' for n in names]

bars = ax.barh(range(len(sorted_ind)), accs, xerr=stds, capsize=3, color=colors, edgecolor='black')
ax.set_yticks(range(len(sorted_ind)))
ax.set_yticklabels(names)
ax.set_xlabel('Cross-Validation Accuracy')
ax.set_title('Individual Feature Type Performance')
ax.set_xlim(0, 1)

# Add accuracy values
for i, (bar, acc) in enumerate(zip(bars, accs)):
    ax.text(acc + 0.02, bar.get_y() + bar.get_height()/2, 
            f'{acc:.1%}', va='center', fontsize=9)

# Legend for colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', edgecolor='black', label='In SRC (production)'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='Experimental only')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('../docs/individual_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary & Recommendations

In [ ]:
print("=" * 70)
print("📋 FEATURE ENGINEERING SUMMARY")
print("=" * 70)
print()

print("📊 FEATURE DIMENSIONS:")
print(f"   Base keypoints: 84 (21 landmarks × 4)")
print(f"   SRC production features: {engineer.get_output_dim()}")
exp_dim = prepare_features(sequences[:1], extract_experimental_all).shape[1]
print(f"   All experimental features: {exp_dim}")
print()

print("🔍 FEATURE SET COMPARISON:")
for r in sorted(results, key=lambda x: x['mean_acc'], reverse=True)[:3]:
    print(f"   • {r['name']}: {r['mean_acc']:.1%}")
print()

print("📈 INDIVIDUAL FEATURE RANKINGS (Top 5):")
for i, r in enumerate(sorted(individual_results, key=lambda x: x['mean_acc'], reverse=True)[:5], 1):
    in_src = "(SRC)" in r['name'] or r['name'] in ['Relative Positions', 'Velocity', 'Inter-finger Distances']
    marker = "✓" if in_src else "✗"
    print(f"   {i}. {r['name']}: {r['mean_acc']:.1%} [{marker} in production]")
print()

print("💡 RECOMMENDATIONS:")
print("   1. SRC features provide good baseline performance")
print("   2. Consider adding top experimental features if accuracy is critical")
print("   3. Monitor for overfitting when adding many features")
print("   4. Run full model training in 04_model_exploration.ipynb")
print("=" * 70)

## Next Steps

Continue to:
- **04_model_exploration.ipynb** - Train and compare different model architectures with the best feature set